# VLM 표 추출 비교 — qwen3-vl 8B vs GLM-OCR vs PaddleOCR-VL (Colab T4)

운영 노드와 같은 T4에서 세 모델을 같은 10쪽에 돌려 **표 품질**과 **페이지당 지연**을 같이 본다.

전제
- 런타임 → 런타임 유형 변경 → **T4 GPU**
- 비교할 약관 PDF 한 건(전체). 3번 셀에서 페이지를 고르고 4번 셀이 10쪽으로 잘라 준다
- 소요: qwen 기준선 10~15분, GLM·Paddle 각 1~2분, vLLM 설치 5~10분. 전체 1시간 안쪽

순서대로 실행한다. 산출물(PDF·캐시·결과·비교 마크다운)은 Drive `MyDrive/bench`에 남아 세션이 끊겨도 유지된다.


In [ ]:
# 1. GPU 확인 · Drive 마운트 · 벤치 데이터 위치
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import os, sys
print("python", sys.version.split()[0])

from google.colab import drive
drive.mount('/content/drive')

BENCH_DATA = '/content/drive/MyDrive/bench'
os.makedirs(f'{BENCH_DATA}/.pdfs', exist_ok=True)
os.environ['BENCH_DATA'] = BENCH_DATA          # bench.py가 캐시·결과를 여기 둔다
print("BENCH_DATA =", BENCH_DATA)


In [ ]:
# 2. 코드 받기 · 의존 설치
%cd /content
!rm -rf Policy-Chunker && git clone -q https://github.com/SM17-YoonDongJu/Policy-Chunker
%cd /content/Policy-Chunker

import sys
if sys.version_info >= (3, 12):
    !pip install -q -e .
else:
    # pyproject가 3.12 이상을 요구한다. 낮으면 의존만 깔고 레포 루트에서 모듈로 실행한다(bench.py는 그렇게 부른다)
    print("Python < 3.12 — 의존만 설치")
    !pip install -q pymupdf pdfplumber pyyaml requests more-itertools numpy


In [ ]:
# 3. 전체 PDF 업로드 → 페이지 분류 (어느 쪽이 PyMuPDF 표 / gap / 본문인지)
from google.colab import files
import shutil, pymupdf

up = files.upload()                      # 약관 PDF 한 건
SRC = f'{BENCH_DATA}/.pdfs/source.pdf'
shutil.move(next(iter(up)), SRC)

from insurance_chunker import extractor
doc = pymupdf.open(SRC)
page_numbers = list(range(1, doc.page_count + 1))
sources, gap, info = extractor.extract_cpu_tables(SRC, page_numbers, use_vision=True)

pymupdf_pages = sorted(sources['pymupdf'])
plumber_only  = sorted(set(sources.get('pdfplumber', {})) - set(pymupdf_pages))
plain         = [p for p in page_numbers if p not in pymupdf_pages and p not in gap and p not in plumber_only]
# 긴 표 후보: PyMuPDF 마크다운 줄 수가 많은 페이지
longest = sorted(pymupdf_pages, key=lambda p: -sources['pymupdf'][p].count('\n'))[:5]

print(f"전체 {doc.page_count}쪽")
print(f"PyMuPDF 표 페이지 {len(pymupdf_pages)}쪽: {pymupdf_pages[:40]}{' …' if len(pymupdf_pages) > 40 else ''}")
print(f"gap 페이지(괘선 없는 표·이미지 표 후보) {len(gap)}쪽: {gap[:40]}")
print(f"pdfplumber만 읽은 페이지 {len(plumber_only)}쪽: {plumber_only[:20]}")
print(f"긴 표 후보: {longest}")
print(f"표 없는 본문 페이지 예: {plain[:10]}")

# 제안: 괘선 표 4 + gap 3 + 긴 표 2 + 본문 1. 다음 셀에서 손으로 고쳐도 된다
PROPOSED = (pymupdf_pages[:4] + gap[:3] + [p for p in longest if p not in pymupdf_pages[:4]][:2] + plain[:1])
print("\n제안 PAGES =", PROPOSED)


In [ ]:
# 4. 10쪽 추출 · 눈으로 볼 PNG 렌더
PAGES = PROPOSED            # ← 필요하면 직접 [12, 45, 46, ...] 로 바꾼다 (원본 1-based)

TRIAL = f'{BENCH_DATA}/.pdfs/vlm-trial-10p.pdf'
doc = pymupdf.open(SRC)
doc.select([p - 1 for p in PAGES])
doc.save(TRIAL)
doc.close()

PNG_DIR = f'{BENCH_DATA}/trial-pages'
os.makedirs(PNG_DIR, exist_ok=True)
trial = pymupdf.open(TRIAL)
for i, page in enumerate(trial, 1):
    page.get_pixmap(dpi=110).save(f'{PNG_DIR}/p{i:02d}.png')
print("새 페이지 → 원본 페이지:", {i: p for i, p in enumerate(PAGES, 1)})
print("저장:", TRIAL)


In [ ]:
# 5. 문서셋 · 변형 yaml 작성 (clone한 레포의 eval/bench 아래)
import yaml
from pathlib import Path

Path('eval/bench/docsets/vlm-trial.yaml').write_text(yaml.safe_dump({
    'name': 'vlm-trial',
    'documents': [{'id': 'vlm-trial-10p', 'pdf': TRIAL, 'type': 'table',
                   'doc_type': 'policy_terms', 'insurer': '미상'}],
}, allow_unicode=True), encoding='utf-8')

COMMON = {'VLM_BACKEND': 'local', 'VLM_TARGET': 'both',   # both: PyMuPDF가 읽은 페이지에도 VLM을 보낸다
          'VISION_MAX_PAGES': '10', 'VLM_CONCURRENCY': '1', 'VLM_TIMEOUT': '600'}
VARIANTS = {
    'vlm-trial-qwen':   {'VLM_URL': 'http://localhost:11434', 'VLM_MODEL': 'qwen3-vl:8b-instruct'},
    'vlm-trial-glm':    {'VLM_URL': 'http://localhost:11434', 'VLM_MODEL': 'glm-ocr:latest',
                         'VLM_PROMPT': 'Table Recognition:'},
    'vlm-trial-paddle': {'VLM_URL': 'http://localhost:8000',  'VLM_MODEL': 'PaddleOCR-VL-1.5-0.9B',
                         'VLM_PROMPT': 'Table Recognition:'},
}
for slug, env in VARIANTS.items():
    Path(f'eval/bench/variants/{slug}.yaml').write_text(yaml.safe_dump(
        {'slug': slug, 'ref': 'WORKTREE', 'env': {**COMMON, **env}, 'status': 'draft'},
        allow_unicode=True), encoding='utf-8')
!ls eval/bench/variants/vlm-trial-*.yaml


In [ ]:
# 6. Ollama 설치 · 기동 · 모델 받기 (qwen 6GB + glm 2.7GB)
# Colab 이미지에는 zstd가 없어 설치 스크립트가 중단된다 — 먼저 넣는다
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import shutil, subprocess, time, requests
OLLAMA = shutil.which('ollama') or '/usr/local/bin/ollama'
assert os.path.exists(OLLAMA), "ollama 바이너리가 없다 — 위 설치 로그를 확인"

subprocess.Popen([OLLAMA, 'serve'], stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get('http://localhost:11434/api/tags', timeout=2); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama serve가 60초 안에 뜨지 않았다:\n" + open('/tmp/ollama.log').read()[-2000:])

!ollama pull qwen3-vl:8b-instruct
!ollama pull glm-ocr:latest
!ollama list


In [ ]:
# 7. 기준선: qwen3-vl 8B  (T4에서 10~15분)
import json
def show(slug):
    r = json.load(open(f'{BENCH_DATA}/results/{slug}/vlm-trial/vlm-trial-10p.json'))
    v, c = r['stages']['vlm'], r['stages']['combine']
    print(f"[{slug}] VLM 호출 {v.get('calls')} / 표 {v.get('tables')} / p50 {v.get('latency_p50')}s / "
          f"p95 {v.get('latency_p95')}s / 잘림 {v.get('truncated')} / 결과 {v.get('outcomes')}")
    print(f"           채택 소스 {c.get('chosen')}  (VLM 단계 {v['seconds']}s)")

!python eval/bench/bench.py run --docset vlm-trial --variant vlm-trial-qwen --to combine
!ollama ps        # PROCESSOR 열이 100% GPU 여야 한다. CPU가 섞이면 수치가 무의미하다
show('vlm-trial-qwen')


In [ ]:
# 8. GLM-OCR (Ollama)
!python eval/bench/bench.py run --docset vlm-trial --variant vlm-trial-glm --to combine
!ollama ps
show('vlm-trial-glm')


In [ ]:
# 9. Ollama 모델 내리고 vLLM 설치 (5~10분. torch가 교체된다 — Ollama 단계를 먼저 끝낸 뒤 실행)
!ollama stop qwen3-vl:8b-instruct; ollama stop glm-ocr:latest
!nvidia-smi --query-gpu=memory.used --format=csv
!pip install -q uv
!uv pip install --system -q vllm --torch-backend=auto     # 드라이버에 맞는 CUDA 빌드를 고른다
# Colab에 원래 있던 torchaudio는 교체된 torch와 CUDA 버전이 달라 transformers import를 죽인다. 표 추출엔 불필요하므로 지운다
!pip uninstall -y -q torchaudio
!python -c "import torch, torchvision; print('torch', torch.__version__, 'cuda', torch.version.cuda, '/ torchvision', torchvision.__version__)"
!python -c "import vllm, transformers; print('vllm', vllm.__version__, '/ transformers', transformers.__version__)"


In [ ]:
# 10. PaddleOCR-VL을 vLLM으로 띄운다 — T4에서 뜨는지 자체가 확인 대상
PADDLE_MODEL = 'PaddlePaddle/PaddleOCR-VL-1.5'     # 1.6의 HF ID가 확인되면 바꾼다
SERVED = 'PaddleOCR-VL-1.5-0.9B'                   # 5번 셀 변형의 VLM_MODEL과 같아야 한다

cmd = (f"vllm serve {PADDLE_MODEL} --trust-remote-code --dtype half "
       f"--max-model-len 8192 --max-num-batched-tokens 16384 "
       f"--no-enable-prefix-caching --mm-processor-cache-gb 0 "
       f"--gpu-memory-utilization 0.5 --served-model-name {SERVED} --port 8000")
vllm_proc = subprocess.Popen(cmd.split(), stdout=open('/tmp/vllm.log', 'w'), stderr=subprocess.STDOUT)

ok = False
for _ in range(180):                                # 최대 15분(모델 다운로드 포함)
    if vllm_proc.poll() is not None:
        break
    try:
        if requests.get('http://localhost:8000/v1/models', timeout=2).ok:
            ok = True; break
    except Exception:
        pass
    time.sleep(5)
if ok:
    print("vLLM 준비됨")
    # 한 장 스모크 — 응답 형식(HTML/OTSL/마크다운)만 확인
    import base64
    pix = pymupdf.open(TRIAL)[0].get_pixmap(dpi=150)
    img = base64.b64encode(pix.tobytes('png')).decode()
    resp = requests.post('http://localhost:8000/v1/chat/completions', json={
        'model': SERVED, 'temperature': 0, 'max_tokens': 2048,
        'messages': [{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{img}'}},
            {'type': 'text', 'text': 'Table Recognition:'}]}]}, timeout=300)
    print(resp.json()['choices'][0]['message']['content'][:600])
else:
    print("vLLM이 뜨지 않았다 — 로그 끝부분:")
    print(open('/tmp/vllm.log').read()[-3000:])


In [ ]:
# 11. PaddleOCR-VL (vLLM)
!python eval/bench/bench.py run --docset vlm-trial --variant vlm-trial-paddle --to combine
show('vlm-trial-paddle')


## 12. (선택) vLLM이 T4에서 안 뜨면 — llama.cpp 폴백

10번 셀이 실패했을 때만 실행한다. CUDA 빌드가 10~15분 걸린다. 운영에서도 vLLM이 안 되면 이 경로를 쓰게 되므로 지연 수치가 그대로 의미 있다.


In [ ]:
# 12. llama.cpp 빌드 · PaddleOCR-VL GGUF 서빙 · 벤치
try:
    vllm_proc.kill()
except Exception:
    pass
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=ON > /dev/null
!cmake --build /content/llama.cpp/build --config Release -j --target llama-server > /dev/null

llama_proc = subprocess.Popen(
    ['/content/llama.cpp/build/bin/llama-server', '-hf', 'PaddlePaddle/PaddleOCR-VL-1.5-GGUF',
     '--port', '8111', '--temp', '0', '-ngl', '99'],
    stdout=open('/tmp/llama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        if requests.get('http://localhost:8111/health', timeout=2).ok: break
    except Exception:
        time.sleep(5)

Path('eval/bench/variants/vlm-trial-paddle-llama.yaml').write_text(yaml.safe_dump(
    {'slug': 'vlm-trial-paddle-llama', 'ref': 'WORKTREE', 'status': 'draft',
     'env': {**COMMON, 'VLM_URL': 'http://localhost:8111', 'VLM_MODEL': '',   # llama-server는 model 필드를 안 받는다
             'VLM_PROMPT': 'Table Recognition:'}}, allow_unicode=True), encoding='utf-8')
!python eval/bench/bench.py run --docset vlm-trial --variant vlm-trial-paddle-llama --to combine
show('vlm-trial-paddle-llama')


In [ ]:
# 13. 페이지별 나란히 비교 — 마크다운 파일 + 노트북 표시
import glob, re
from IPython.display import display, Markdown, Image
from insurance_chunker.models import compute_doc_hash

doc_dir = Path(BENCH_DATA) / '.artifacts' / compute_doc_hash(TRIAL)[:16]
latest = lambda pat: max(glob.glob(str(doc_dir / pat)), key=os.path.getmtime)
cpu = json.load(open(latest('tables_cpu-*.json')))['output']['sources']
pymu, plumber = cpu.get('pymupdf', {}), cpu.get('pdfplumber', {})

vlms = {}                                   # 라벨 → {"tables": {page: md}, "m": metrics}
for f in sorted(glob.glob(str(doc_dir / 'vlm-*.json')), key=os.path.getmtime):
    d = json.load(open(f))
    cfg = d.get('config', {})
    if cfg.get('skipped'):
        continue
    label = cfg.get('model') or f"llama-server({cfg.get('prompt')})"
    vlms[label] = {'tables': d['output'], 'm': d['record']['metrics']}

def cells(mdtxt):
    out = set()
    for ln in (mdtxt or '').splitlines():
        if '|' in ln and not re.fullmatch(r'[\s|:-]+', ln):
            out |= {c.strip() for c in ln.strip().strip('|').split('|') if c.strip()}
    return out

lines = ['# VLM 표 추출 비교 (10쪽, T4)', '', '| 모델 | 호출 | 표 | p50(s) | p95(s) | 잘림 |', '|---|---|---|---|---|---|']
for lb, v in vlms.items():
    m = v['m']
    lines.append(f"| {lb} | {m.get('calls')} | {m.get('tables')} | {m.get('latency_p50')} | {m.get('latency_p95')} | {m.get('truncated')} |")
lines += ['', '셀 일치율 = (VLM 셀 ∩ PyMuPDF 셀) / PyMuPDF 셀. 괘선 표 페이지에서만 의미가 있다. 숫자·코드 오독은 직접 확인.', '']

for i in range(1, len(PAGES) + 1):
    p = str(i)
    ref = cells(pymu.get(p))
    lines += [f'## p{i} (원본 p{PAGES[i-1]})', '', f'![p{i}](trial-pages/p{i:02d}.png)', '']
    lines += ['### PyMuPDF', '', pymu.get(p) or '(없음)', '']
    if plumber.get(p):
        lines += ['### pdfplumber', '', plumber[p], '']
    for lb, v in vlms.items():
        mdtxt = v['tables'].get(p)
        rate = f" — 셀 일치율 {len(cells(mdtxt) & ref) / len(ref):.2f}" if ref and mdtxt else ''
        lines += [f'### {lb}{rate}', '', mdtxt or '(출력 없음)', '']

out = Path(BENCH_DATA) / 'vlm-trial-compare.md'
out.write_text('\n'.join(lines), encoding='utf-8')
print("저장:", out)

for i in range(1, len(PAGES) + 1):
    display(Image(f'{PNG_DIR}/p{i:02d}.png', width=520))
    sec = '\n'.join(lines)
    start = sec.index(f'## p{i} ')
    end = sec.find(f'## p{i+1} ', start)
    display(Markdown(sec[start:end if end > 0 else None]))


## 판정 기준

| 페이지 종류 | 보는 것 |
|---|---|
| 괘선 표 (PyMuPDF가 읽은 쪽) | 셀 일치율과, 숫자·질병코드(I00 ↔ 100)·한자 오독. 하나라도 틀리면 치명적 |
| gap (괘선 없는 표·이미지 표) | 이미지와 행·열 수가 같은가. 병합셀이 값 반복으로 풀렸는가 |
| 긴 표 | `잘림` 수. finish_reason=length로 끊긴 표는 시간도 품질도 잃는다 |
| 본문만 있는 페이지 | 출력이 없어야 한다. 없는 표를 만들면 감점 |
| 지연 | p50·p95. 운영 qwen3-vl 8B는 p50 88.6초였다 |

`[?]` 셀 수는 우리 프롬프트 전용 표식이라 Paddle·GLM에는 나오지 않는다. 그 항목은 비교에서 무시한다.

이긴 모델 하나만 클러스터로 올린다. 그때 유의할 것은 vLLM이 VRAM 85%를 잡고 있어 Ollama 몫이 2.4GB뿐이라는 점이다. GLM-OCR을 Ollama 파드에 더하려면 K8s-manifests에서 `vllm-qwen3vl`을 내리거나 메모리 비율을 낮추는 PR이 먼저다.
